# 02 — Data Preparation

Reads raw parquet files produced by CarpDataStream, applies cleaning, and writes a prepared dataset to `data/processed/`.

**Steps:**
1. Filter all sensor streams to the conference week (Oct 13–16, inclusive).
2. Convert cumulative step counts to per-interval deltas.
3. Parse surveys into structured tables (onboarding, daily stress/sleep, WHO-5, BFI-10).
4. Write everything to `data/processed/` as parquet files.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd

# ── Paths ──
PROJECT_ROOT = Path.cwd().parent
RAW_PARQUET = PROJECT_ROOT / "data" / "raw" / "parquet"
OUT_DIR = PROJECT_ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from data.surveys import (
    load_submitted,
    parse_daily_sleep,
    parse_daily_stress,
    parse_onboarding,
    parse_who5,
    parse_bfi10,
    parse_event_ratings,
)

print(f"Raw parquets: {RAW_PARQUET}")
print(f"Output dir:   {OUT_DIR}")
print(f"Available:    {sorted(f.stem for f in RAW_PARQUET.glob('*.parquet'))}")

Raw parquets: /home/miki/Study/4_forth/UbiLife/data/raw/parquet
Output dir:   /home/miki/Study/4_forth/UbiLife/data/processed
Available:    ['activity', 'airquality', 'ambientlight', 'batterystate', 'completedapptask', 'deviceinformation', 'error', 'heartbeat', 'image', 'location', 'screenevent', 'stepcount', 'survey', 'triggeredtask', 'weather']


## 1. Filter Sensor Streams to Conference Week (Oct 13–16)

Discard all sensor data outside the main conference dates to focus analysis on the event itself.

In [2]:
CONF_START = pd.Timestamp("2025-10-13")
CONF_END = pd.Timestamp("2025-10-16 23:59:59.999999")

SENSOR_STREAMS = [
    "activity", "airquality", "ambientlight", "batterystate",
    "heartbeat", "location", "screenevent", "stepcount", "weather",
]


def load_raw(name: str) -> pd.DataFrame:
    """Load a raw parquet and add timestamp + participant_id columns."""
    df = pd.read_parquet(RAW_PARQUET / f"{name}.parquet")
    df["timestamp"] = pd.to_datetime(
        df["measurement.sensorStartTime"], unit="us", errors="coerce"
    )
    df["participant_id"] = df["dataStream.studyDeploymentId"]
    return df


def filter_conference_week(df: pd.DataFrame) -> pd.DataFrame:
    """Keep only rows within Oct 13-16 2025 (inclusive)."""
    mask = (df["timestamp"] >= CONF_START) & (df["timestamp"] <= CONF_END)
    return df.loc[mask].copy()


results = []
for name in SENSOR_STREAMS:
    src = RAW_PARQUET / f"{name}.parquet"
    if not src.exists():
        print(f"SKIP {name} (not found)")
        continue

    df = load_raw(name)
    n_before = len(df)
    df = filter_conference_week(df)
    n_after = len(df)

    # Save filtered (stepcount handled separately in next cell)
    if name != "stepcount":
        df.to_parquet(OUT_DIR / f"{name}.parquet", index=False)
    else:
        df_steps = df  # keep for next cell

    results.append({
        "stream": name,
        "before": n_before,
        "after": n_after,
        "kept_pct": n_after / n_before * 100,
        "participants": df["participant_id"].nunique(),
    })

filter_df = pd.DataFrame(results)
filter_df["kept_pct"] = filter_df["kept_pct"].round(1)
filter_df

,stream,before,after,kept_pct,participants
0,activity,199336,166530,83.5,77
1,airquality,3837,3057,79.7,65
2,ambientlight,3115,1999,64.2,19
3,batterystate,18119,13284,73.3,58
4,heartbeat,204811,107994,52.7,80
5,location,133082,93913,70.6,78
6,screenevent,5657,5606,99.1,22
7,stepcount,447625,393319,87.9,77
8,weather,3854,3074,79.8,65


## 2. Convert Cumulative Step Counts to Deltas

The raw step count is a monotonically increasing pedometer counter. We compute per-interval deltas (diff), clamping negative values (pedometer resets) to zero.

In [3]:
df_steps = df_steps.sort_values(["participant_id", "timestamp"])

df_steps["steps_cumulative"] = df_steps["measurement.data.steps"]
df_steps["steps_delta"] = (
    df_steps.groupby("participant_id")["steps_cumulative"]
    .diff()
    .clip(lower=0)
)
df_steps["steps_delta"] = df_steps["steps_delta"].fillna(0).astype(int)

df_steps.to_parquet(OUT_DIR / "stepcount.parquet", index=False)

# Show a sample
pid = df_steps["participant_id"].value_counts().index[0]
sample = df_steps[df_steps["participant_id"] == pid].head(10)
print(f"Sample deltas for participant {pid[:12]}...")
sample[["timestamp", "steps_cumulative", "steps_delta"]]

Sample deltas for participant b403e2f9-0be...


,timestamp,steps_cumulative,steps_delta
69912,2025-10-13 05:07:23.657356,15375,0
69913,2025-10-13 05:07:24.057580,15376,1
69914,2025-10-13 05:07:24.544699,15377,1
69915,2025-10-13 05:09:04.068720,15384,7
69916,2025-10-13 05:09:04.390789,15385,1
69917,2025-10-13 05:09:05.034738,15386,1
69918,2025-10-13 05:09:17.826594,15407,21
69919,2025-10-13 05:09:22.004190,15408,1
69920,2025-10-13 05:09:22.008288,15413,5
69921,2025-10-13 05:09:57.967036,15428,15


In [4]:

# Aggregate step deltas per participant per hour
df_steps["hour"] = df_steps["timestamp"].dt.floor("h")

stepcount_hourly = (
    df_steps.groupby(["participant_id", "hour"])["steps_delta"]
    .sum()
    .reset_index(name="steps")
)

stepcount_hourly.to_parquet(OUT_DIR / "stepcount_hourly.parquet", index=False)

print(f"Hourly step counts: {len(stepcount_hourly)} rows, "
      f"{stepcount_hourly['participant_id'].nunique()} participants")
stepcount_hourly.head(10)


Hourly step counts: 1299 rows, 77 participants


,participant_id,hour,steps
0,02afb169-db56-4dc6-a16d-a901e2d24b9a,2025-10-14 18:00:00,378
1,02afb169-db56-4dc6-a16d-a901e2d24b9a,2025-10-14 19:00:00,920
2,02afb169-db56-4dc6-a16d-a901e2d24b9a,2025-10-15 06:00:00,355
3,02afb169-db56-4dc6-a16d-a901e2d24b9a,2025-10-15 07:00:00,1170
4,02afb169-db56-4dc6-a16d-a901e2d24b9a,2025-10-15 08:00:00,598
5,02afb169-db56-4dc6-a16d-a901e2d24b9a,2025-10-15 09:00:00,422
6,02afb169-db56-4dc6-a16d-a901e2d24b9a,2025-10-15 10:00:00,436
7,02afb169-db56-4dc6-a16d-a901e2d24b9a,2025-10-15 11:00:00,626
8,02afb169-db56-4dc6-a16d-a901e2d24b9a,2025-10-15 12:00:00,533
9,02afb169-db56-4dc6-a16d-a901e2d24b9a,2025-10-15 13:00:00,150


## 3. Parse Surveys into Structured Tables

Each survey type gets its own clean DataFrame:

| Table | Shape | Key columns |
|---|---|---|
| `survey_onboarding` | 1 row/participant | age, gender, academic role, presenter types |
| `survey_daily_stress` | 1 row/participant/day | stress_level (0–10) |
| `survey_daily_sleep` | 1 row/participant/day | sleep quality + hours |
| `survey_who5` | 1 row/participant | q1–q5, raw_sum, pct_score |
| `survey_bfi10` | 1 row/participant | Big Five dimensions |
| `survey_event_ratings` | 1 row/participant/event | session (A–H) and keynote (1–3) ratings |

In [5]:
sub = load_submitted(RAW_PARQUET)

survey_parsers = {
    "survey_daily_stress": parse_daily_stress,
    "survey_daily_sleep": parse_daily_sleep,
    "survey_onboarding": parse_onboarding,
    "survey_who5": parse_who5,
    "survey_bfi10": parse_bfi10,
    "survey_event_ratings": parse_event_ratings,
}

survey_tables = {}
summary = []

for table_name, parser in survey_parsers.items():
    df = parser(sub)
    df.to_parquet(OUT_DIR / f"{table_name}.parquet", index=False)
    survey_tables[table_name] = df
    summary.append({
        "table": table_name,
        "rows": len(df),
        "participants": df["participant_id"].nunique(),
        "columns": ", ".join(c for c in df.columns if c not in ["participant_id", "timestamp", "date"]),
    })

pd.DataFrame(summary)

,table,rows,participants,columns
0,survey_daily_stress,81,36,stress_level
1,survey_daily_sleep,73,36,"sleep_quality_value, sleep_quality_text, sleep..."
2,survey_onboarding,38,38,"age, gender_value, gender_text, academic_value..."
3,survey_who5,37,37,"q1, q2, q3, q4, q5, raw_sum, pct_score"
4,survey_bfi10,42,42,"extraversion, agreeableness, conscientiousness..."
5,survey_event_ratings,224,38,"attending_text, rating_value, rating_text, eve..."


In [6]:
for name, df in survey_tables.items():
    print(f"── {name} ──")
    display(df.head())
    print()

── survey_daily_stress ──


,participant_id,date,timestamp,stress_level
7,051da686-748f-4343-bd85-870f579bd485,2025-10-16,2025-10-16 15:20:20.140230,7.0
26,b403e2f9-0beb-4648-816d-95181eed17bd,2025-10-14,2025-10-14 19:40:19.061501,5.0
27,b403e2f9-0beb-4648-816d-95181eed17bd,2025-10-14,2025-10-14 19:40:19.061501,5.0
46,52ba4418-137f-4e45-9089-ca5f71783ce8,2025-10-14,2025-10-14 13:00:20.709400,4.0
47,aaa33240-0619-4dff-91fd-e36e2ad08c14,2025-10-14,2025-10-14 13:00:10.522427,1.0



── survey_daily_sleep ──


,participant_id,date,timestamp,sleep_quality_value,sleep_quality_text,sleep_hours_value,sleep_hours_text
76,72405115-4aec-4a56-b78e-ca21fd75dc18,2025-10-16,2025-10-16 07:38:00.954374,2,Neutral,3,<6 hours
91,f7545e44-04d8-4daf-9e16-805b5913b801,2025-10-14,2025-10-14 15:13:50.809450,3,Good,2,6-8 hours
93,051da686-748f-4343-bd85-870f579bd485,2025-10-14,2025-10-14 15:22:11.102862,0,Very Bad,3,<6 hours
111,9a10f570-ea36-47c7-abd5-447d1a165784,2025-10-16,2025-10-16 07:43:42.690059,3,Good,2,6-8 hours
112,9a10f570-ea36-47c7-abd5-447d1a165784,2025-10-16,2025-10-16 07:43:42.690059,3,Good,2,6-8 hours



── survey_onboarding ──


...

,participant_id,timestamp,q1,q2,q3,q4,q5,raw_sum,pct_score
86,9a10f570-ea36-47c7-abd5-447d1a165784,2025-10-10 10:44:14.806189,4,3,4,4,4,19,76
121,f7545e44-04d8-4daf-9e16-805b5913b801,2025-10-12 15:49:49.794689,2,3,4,2,4,15,60
142,e7cb43d0-f5ac-446d-932e-31c349076dd5,2025-10-13 09:33:20.699566,3,3,4,2,4,16,64
153,051da686-748f-4343-bd85-870f579bd485,2025-10-13 11:03:20.845027,2,2,1,0,2,7,28
157,04e8fb00-9133-44e9-8c8a-558bf79e6c4e,2025-10-13 11:39:53.520289,2,2,2,2,3,11,44



── survey_bfi10 ──


,participant_id,timestamp,extraversion,agreeableness,conscientiousness,neuroticism,openness
87,9a10f570-ea36-47c7-abd5-447d1a165784,2025-10-10 10:46:01.903186,1.5,2.5,4.0,1.5,1.5
120,f7545e44-04d8-4daf-9e16-805b5913b801,2025-10-12 15:43:20.484940,2.0,3.0,3.0,2.0,3.5
136,e7cb43d0-f5ac-446d-932e-31c349076dd5,2025-10-13 09:32:48.716175,2.5,2.5,2.5,2.0,2.0
147,f226f897-a935-482c-930a-2f83a6cc6423,2025-10-13 10:04:36.128738,1.5,3.0,2.0,2.0,3.0
152,051da686-748f-4343-bd85-870f579bd485,2025-10-13 11:02:35.732382,1.5,2.0,3.0,4.0,2.5



── survey_event_ratings ──


,participant_id,timestamp,attending_text,rating_value,rating_text,event_type,event_id
0,9a10f570-ea36-47c7-abd5-447d1a165784,2025-10-14 11:06:10.916310,A3: Mental Health,3,Interesting,session,A
1,9a10f570-ea36-47c7-abd5-447d1a165784,2025-10-14 11:06:10.916310,A3: Mental Health,3,Interesting,session,A
2,e7cb43d0-f5ac-446d-932e-31c349076dd5,2025-10-14 11:31:24.280974,A3: Mental Health,3,Interesting,session,A
3,b403e2f9-0beb-4648-816d-95181eed17bd,2025-10-14 19:41:42.686959,A3: Mental Health,3,Interesting,session,A
4,b403e2f9-0beb-4648-816d-95181eed17bd,2025-10-14 19:41:42.686959,A3: Mental Health,3,Interesting,session,A


## 4. Summary

In [7]:
all_files = sorted(OUT_DIR.glob("*.parquet"))

file_summary = []
for f in all_files:
    df = pd.read_parquet(f)
    file_summary.append({
        "file": f.name,
        "rows": len(df),
        "columns": len(df.columns),
        "size_kb": f.stat().st_size // 1024,
    })

summary_df = pd.DataFrame(file_summary)
print(f"Output directory: {OUT_DIR}")
print(f"Total files: {len(all_files)}")
print(f"Total size: {summary_df['size_kb'].sum():,} KB")
print()
summary_df

Output directory: /home/miki/Study/4_forth/UbiLife/data/processed
Total files: 16
Total size: 22,096 KB



,file,rows,columns,size_kb
0,activity.parquet,166530,17,3066
1,airquality.parquet,3057,21,100
2,ambientlight.parquet,1999,20,95
3,batterystate.parquet,13284,17,307
4,heartbeat.parquet,107994,18,1796
5,location.parquet,93913,28,7768
6,screenevent.parquet,5606,16,133
7,stepcount.parquet,393319,18,8609
8,stepcount_hourly.parquet,1299,3,12
9,survey_bfi10.parquet,42,7,7
